# 05: Run your circuit on a remote CPU

**What you will learn:** a cloud job runs elsewhere and has its own ID, status and result. The circuit is still the same FlagQuantum circuit.

We will send one asymmetric circuit to Jiuding. This is a CPU experiment, not an accelerator or speedup demonstration.

**Learning goal:** submit one CPU job, recover its receipt and compare the remote state with the local state.

**Prerequisites:** lesson 01. Real execution additionally needs a Jiuding account, verified image/project/queue and shared storage. Local rehearsal alone does not test the service.

Choose a route in the [workshop guide](../README.md).


## 1. Prepare the workshop

Run this setup cell unchanged. It finds your personal result directory. The event environment supplies credentials and the job image; no secrets belong in notebook cells. See [Setup](../SETUP.md) when using your own computer.


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
WORKSHOP = next((root / "flagos2026" for root in search_roots
                 if (root / "flagos2026/notebooks").is_dir()), None)
if WORKSHOP is None:
    raise RuntimeError("Cannot locate flagos2026/notebooks. Open the deployed workshop first.")
sys.path.insert(0, str(WORKSHOP / "scripts"))
from cloud_lesson import workshop_directory

WORKSHOP = workshop_directory(Path.cwd())
repository = WORKSHOP.parent.parent
ROOT = repository if (repository / "pyproject.toml").is_file() else WORKSHOP
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)


## 2. Build an asymmetric circuit and predict the answer

Apply X only to q0 in a three-qubit circuit: the ideal outcome is `100` with probability one. First check this locally.

Each horizontal line in the circuit diagram is a qubit; gates execute from left to right. Vertical wire order is not the bit-string display order on a cloud platform.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import flagquantum as fq

circuit = fq.Circuit(3).x(0)
fig, ax = circuit.draw(format="mpl", show_all_qubits=True)
plt.show()
local = fq.run(circuit)
labels = [format(index, "03b") for index in range(8)]
print("Local probabilities (q0 q1 q2):", dict(zip(labels, local.state.reshape(-1).abs().square().tolist())))


## 3. Choose the remote CPU

The image supplies the installed software. The event environment preconfigures it; at home use your own Jiuding image, project and queue. **Enabling the switch and running the submission cell creates a real CPU job and consumes queue resources.**

**Concept:** the job runs in another container, not this notebook kernel. The image determines its installed software and versions; shared storage exchanges scripts and results. The personal workspace is resolved during setup, and the actual configuration is printed below.

**Expected output:** a CPU statevector matching the local asymmetric circuit (q0=1, hence 100). `Restored job` means the saved task ID was recovered, not executed again.

**Common misconception:** `os.getenv` only reads configuration; it does not discover or create an image. Published examples default to local rehearsal (`False`). The event deployment must set the visible switch to `True` after configuring and verifying the image/project/queue; credentials alone do not authorize automatic submission.


In [ ]:
target = "jiuding:cpu"
image = os.getenv("JIUDING_IMAGE", "")
project = os.getenv("JIUDING_PROJECT", "")
queue = os.getenv("JIUDING_QUEUE")
submit_to_cloud = False

print("Job image:", image or "not configured")
print("Project:", project or "not configured")
print("Queue:", queue or "platform default")
print("Result directory:", OUTPUTS)
print("Real submission enabled:", submit_to_cloud)


### Choose the receipt location (run unchanged)

This cell selects a file in your personal persistent result directory and checks for uncertain attempts from earlier workshop versions. The SDK handles saving, configuration checks and matching-request recovery through `receipt=`. Keep the receipt and its SDK-created journal together.

Existing manual receipts need explicit `fq.restore_job(receipt_path)`; they are not silently adopted. Never delete old records to force another submission.


In [ ]:
receipt_path = OUTPUTS / "jiuding-x0-three-qubits.receipt.json"
job = None
if submit_to_cloud:
    if not image or not project:
        raise RuntimeError("Ask your instructor to configure the Jiuding image and project; nothing submitted.")
    if not receipt_path.exists() and (OUTPUTS / "jiuding-x0-three-qubits.submission.json").exists():
        raise RuntimeError("An earlier workshop attempt has no receipt. Reconcile it on the platform; do not resubmit.")
print("Receipt path:", receipt_path.resolve())


## 4. Submit through FlagQuantum

`fq.submit(..., receipt=receipt_path)` submits and saves on the first run, then restores on matching reruns without submitting again. It returns without waiting for execution. Configuration changes, incomplete records and recovery failures raise instead of creating another task; no manual hashing or `job.save()` is needed. A new experiment needs a new receipt path after checking the old attempt.


In [ ]:
job = None
if submit_to_cloud:
    job = fq.submit(
        circuit, target=target, image=image, project=project, queue=queue,
        receipt=receipt_path,
    )
    print("Job ID:", job.id)
else:
    print("Local rehearsal only. Enable the switch to submit a real task.")


## 5. Query progress

This is a single read-only query. `queued` means waiting, `running` means executing, and `succeeded` means the platform reports completion. Result delivery can still be delayed or unavailable.

**Read the output:** `Job ID` identifies the task, not a completed calculation. A matching rerun prints the same ID because the SDK restored it. Retrieve the result before comparing local and remote execution.

**Rerun safely:** repeat the same submission call with the same persistent receipt path; the SDK validates and restores it. To check a queued task, rerun only the result cell. Never delete recovery files to retry an uncertain submission. Changed circuit, target or shots needs a new experiment path after checking the old attempt.


In [ ]:
if job is not None:
    print("Status:", job.status())


## 6. Wait briefly and read the result

`job.wait()` is a convenience for polling status and retrieving the result. A timeout leaves the task running. Rerun this cell later. Alternatively use `job.result()` once the status is `succeeded`. No local result is substituted for a remote failure.


In [ ]:
result = None
if job is not None:
    try:
        result = job.wait(timeout=30)
    except TimeoutError:
        print("Result not available yet; last provider status:", job.raw_status)
        print("Rerun only this result cell later; do not resubmit. If completion is reported but results remain unavailable, ask the instructor to inspect task logs.")
if result is not None:
    import torch

    torch.testing.assert_close(result.state, local.state)
    print("Remote probabilities (q0 q1 q2):", dict(zip(labels, result.state.reshape(-1).abs().square().tolist())))
    print("The remote CPU calculation matches the local calculation.")


## Take it home

The standard API is the same outside Jupyter. The first call submits real work and saves its receipt; matching reruns with the same persistent path restore it. The SDK owns configuration checks and duplicate-submission guards. Configure your image, project and queue first.

```python
import os
import flagquantum as fq

circuit = fq.Circuit(3).x(0)
job = fq.submit(
    circuit, target="jiuding:cpu", image=os.environ["JIUDING_IMAGE"],
    project=os.environ["JIUDING_PROJECT"], queue=os.getenv("JIUDING_QUEUE"),
    receipt="x0-three-qubits.receipt.json",
)
print(job.id, job.status())
```

For a saved task: `job = fq.restore_job("x0-three-qubits.receipt.json")`. When complete, inspect `job.result().state`.

**Explain the result:** which calls submit, query, wait and retrieve? Does remote execution of a tiny circuit prove a speedup? Use [Cloud job field notes](../CLOUD_JOBS.md) for complete Python programs and accelerator choices.
